# Einstein 1905: moving bodies and $E = mc^2$

*astro · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/astro/einstein_1905_relativity)

Einstein's own route through his two 1905 relativity papers. From two postulates — the same laws in every inertial frame, light always at $c$ — a batch of light clocks shows moving time slowed by $\gamma$, rods shortened by $1/\gamma$, simultaneity that depends on the observer and speeds that never add beyond $c$. The mass–energy paper (a body emitting light energy $L$ loses mass $L/c^2$) is checked on the He-4 mass defect and $e^+e^-$ annihilation, and a fit to Frisch and Smith's 1963 muon counts on Mt Washington and at sea level gives $\gamma \approx 9$.

**Vector form**

$$\Delta t = \gamma\,\Delta\tau,\quad L = L_0/\gamma,\quad \gamma = \frac{1}{\sqrt{1 - v^2/c^2}},\qquad w = \frac{u + v}{1 + uv/c^2},\qquad \Delta m = \frac{L}{c^2},\quad E^2 = (pc)^2 + (mc^2)^2$$

**Index (tensor) form**

$$x'^\mu = \Lambda^\mu{}_\nu\,x^\nu,\qquad \eta_{\mu\nu}\Delta x'^\mu\Delta x'^\nu = \eta_{\mu\nu}\Delta x^\mu\Delta x^\nu,\qquad \eta_{\mu\nu}\,p^\mu p^\nu = -m^2c^2,\qquad N_{\text{sea}} = N_{\text{top}}\,e^{-t_{\text{lab}}/(\gamma\tau_\mu)}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{bt},\ y_{bt}$ | `light_clocks(beta, D, t_max, n)` | (B, T) | photon of clock $b$ at time $t$: horizontal speed $\beta_b$, vertical $\sqrt{1-\beta_b^2}$, so always $c$ |
| $\gamma(\beta_b)$ | `1 / torch.sqrt(1 - beta**2)` | (B,) | one tick takes $\gamma\cdot 2D/c$; ticks counted from the zigzag |
| $\Lambda^\mu{}_\nu(\beta_b)$ | `boost(beta)` | (B, 4, 4) | a batch of boosts along $x$, $\eta = \operatorname{diag}(-1, 1, 1, 1)$ |
| $\Lambda^\mu{}_\nu\,x^\nu$ | `einsum('...mn,kn->...km', Lam, x)` | (B, K, 4) | repeated $\nu$ ⇒ summed: every event in every frame; the train's flashes get $\Delta t' = -\gamma\beta\Delta x$ |
| $\eta_{\mu\nu}\Delta x^\mu\Delta x^\nu$ | `einsum('mn,...m,...n->...', eta, dx, dx)` | (…) | the interval, unchanged by every boost |
| $\tfrac{L}{2}\gamma(1\mp\beta\cos\theta)$ | `pulse_energies(L, beta, theta)` | (B, Θ, 2) | the two light pulses seen from the moving frame: together $\gamma L$, so $K_0 - K_1 = (L/c^2)(\gamma - 1)c^2$ |
| $B = Zm_H + Nm_n - M$ | `binding_energy(Z, N, M)` | (K,) | mass defect × 931.494 MeV/u (He-4: 28.30 MeV) |
| $\gamma_{\text{fit}}$ | `muon_gamma(data, dEdx)` | () | Newton steps with autograd on $\chi^2(\gamma)$ for the two muon rates |

This is one of Einstein's four 1905 "miracle year" papers — two of them, in fact: "On the electrodynamics of moving bodies" (June) and "Does the inertia of a body depend upon its energy content?" (September). The others are the light-quantum paper (quantum: Photoelectric effect and light quanta) and Brownian motion (thermodynamics: Random walk and diffusion). For relativity as geometry see Special relativity: Minkowski diagrams (astro), for the field tensor the Lorentz boost of $F^{\mu\nu}$ (em), and for collisions Four-momentum and two-body decay (nuclear). Units $c = 1$, mirror spacing $D = 1$. Einstein's mass argument: a body at rest emits $L/2$ forwards and $L/2$ backwards; in a frame moving at $v$ the light carries $\gamma L$, more than $L$, and energy conservation in both frames forces the body's kinetic energy to drop by $L(\gamma - 1)$, exactly what a mass $L/c^2$ would carry (he kept only the low-speed term $\tfrac12 (L/c^2)v^2$). Atomic masses are AME2020. Real data (data/frisch_smith_1963_muons.csv): D. H. Frisch and J. H. Smith, Am. J. Phys. 31, 342 (1963), doi:10.1119/1.1969508 — muons slowed to the same energy, $563 \pm 10$ per hour at 1917 m on Mt Washington and $408 \pm 9$ per hour at sea level (published measurement values; Wikipedia's summary rounds the latter to 412). The flight takes about 6.4 μs, almost three muon lifetimes of 2.2 μs, so without time dilation only about 30 per hour would arrive; the counts give the muon clock 0.71 μs and $\gamma \approx 9.1 \pm 0.8$ (the paper, with its measured decay curve: $8.8 \pm 0.8$). The prediction from the speeds integrates the slowdown from $\beta \approx 0.995$ to $\approx 0.989$ through the air (ionisation loss $dE/dx$, isothermal atmosphere): about 8.4, as in the paper. Rossi and Hall (1941) first showed the effect with cosmic-ray muons at Echo Lake and Denver.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/astro/einstein_1905_relativity/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    beta_fast, n_clocks, dEdx = params["beta_fast"], params["n_clocks"], params["dEdx"]

    # light clocks: every clock and every instant at once
    beta = tensors.clock_speeds(beta_fast, n_clocks)  # (B,)
    t, x, y = solve.light_clocks(beta, D=1.0, t_max=8.4, n=1681)
    ticks = solve.count_ticks(y)
    for b, k in zip(beta.tolist(), ticks.tolist()):
        g = equation.gamma(torch.tensor(b)).item()
        print(
            f"clock at β = {b:.3f}: γ = {g:.3f}, ticks {k} in t = 8.4 (rest clock: {int(ticks[0])}),"
            f" rod length L/L0 = {solve.contracted_length(1.0, torch.tensor(b)).item():.3f}"
        )
    w = equation.velocity_addition(torch.tensor(0.9), torch.tensor(0.9))
    w_c = equation.velocity_addition(torch.tensor(1.0), torch.tensor(0.9))
    print(f"0.9c ⊕ 0.9c = {w:.4f}c (not 1.8c); c ⊕ 0.9c = {w_c:.1f}c")
    flashes = torch.tensor([[0.0, -1, 0, 0], [0.0, 1, 0, 0]], dtype=torch.float64)  # t = 0 on the platform
    dt = equation.transform(equation.boost(torch.tensor(0.6)), flashes)
    print(f"flashes simultaneous on the platform, 2 light-units apart: Δt′ = {dt[1, 0] - dt[0, 0]:.3f} at 0.6c")

    # E = mc²
    b_grid = torch.linspace(0.01, 0.99, 99, dtype=torch.float64)
    dK = solve.kinetic_energy_drop(1.0, b_grid, torch.zeros_like(b_grid))
    dm = dK / (equation.gamma(b_grid) - 1)
    print(f"1905 mass–energy argument: Δm·c²/L = {dm.min():.12f} … {dm.max():.12f} for β = 0.01 … 0.99")
    names = list(tensors.NUCLIDES)
    Z, N, M = tensors.nuclear_masses(names)
    B = equation.binding_energy(Z, N, M)
    for n, b, a in zip(names, B.tolist(), (Z + N).tolist()):
        print(f"  {n:6s} binding energy {b:9.3f} MeV  ({b / a:.3f} MeV per nucleon)")
    photons = solve.annihilate_at_rest(tensors.M_ELECTRON)
    print(f"e⁺e⁻ at rest → 2 γ of {photons[0, 0]:.5f} MeV each (2 m_e c² = {2 * tensors.M_ELECTRON:.5f} MeV)")

    # Frisch & Smith (1963): muons from Mt Washington to sea level
    data = tensors.frisch_smith()
    mu = solve.muon_gamma(data, dEdx)
    print(
        f"muons: lab flight time {mu['t_lab']:.2f} μs; without dilation {mu['no_dilation_rate']:.0f}/h would arrive,"
        f" measured {data['rate'][1]:.0f} ± {data['err'][1]:.0f}/h"
    )
    print(
        f"  γ from the counts {mu['gamma_fit']:.2f} ± {mu['gamma_err']:.2f} (muon clock: {mu['tau_rest']:.2f} μs);"
        f" γ from the speeds {mu['gamma_pred']:.2f}; published 8.8 ± 0.8 vs 8.4 ± 2"
    )
    print(f"  speed at sea level {mu['beta_bottom'][0]:.4f} … {mu['beta_bottom'][1]:.4f} (published 0.9881 … 0.9897)")
    return plot.animate(beta, t, x, y, (names, B, Z + N), mu, out_dir / "einstein_1905_relativity.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/astro/einstein_1905_relativity/equation.py
r"""Physics → tensor translation.

Einstein, "On the electrodynamics of moving bodies" (1905), from two postulates: (1) the laws of physics are the
same in every inertial frame; (2) light moves at c in every inertial frame. Units with c = 1 unless stated.

A light clock (a photon bouncing between mirrors a distance D apart) moving sideways at speed β: the photon runs
along the diagonal at speed 1, so one tick takes

    \Delta t = \gamma\,\Delta\tau,\qquad \gamma = 1/\sqrt{1 - \beta^2}

and the same argument for a clock lying along the motion gives the contraction L = L_0/γ. Events are 4-vectors
x^μ = (t, x, y, z), the metric is η = diag(−1, +1, +1, +1) and a boost along x is

    x'^\mu = \Lambda^\mu{}_\nu(\beta)\,x^\nu,\qquad \Delta s^2 = \eta_{\mu\nu}\,\Delta x^\mu\Delta x^\nu  (ν, μ summed)

Speeds add as w = (u + v)/(1 + uv), never beyond 1. A body's 4-momentum p^μ = (γm, γmβ, 0, 0) satisfies
η_{μν} p^μ p^ν = −m², i.e. E² = (pc)² + (mc²)².

"Does the inertia of a body depend upon its energy content?" (1905): a body at rest emits light energy L/2 in two
opposite directions. Seen from a frame moving at β the two pulses carry (L/2)γ(1 ∓ β cos θ), together γL, so the
body's kinetic energy drops by K_0 − K_1 = L(γ − 1) — the kinetic energy of a mass L/c²: it lost mass Δm = L/c².
A nucleus weighs less than its parts by its binding energy: B = Z m_H + N m_n − M(Z, N), in u × 931.494 MeV.
"""

import torch

U_MEV = 931.49410372  # MeV per atomic mass unit (CODATA 2022)
M_H = 1.007825031898  # u, hydrogen-1 atom (AME2020): proton + electron
M_N = 1.00866491606  # u, neutron


def gamma(beta: torch.Tensor) -> torch.Tensor:
    """Lorentz factor γ = 1/√(1 − β²), any shape."""
    return 1 / torch.sqrt(1 - beta**2)


def metric(dtype=torch.float64) -> torch.Tensor:
    """η_{μν} = diag(−1, +1, +1, +1): (4, 4)."""
    return torch.diag(torch.tensor([-1.0, 1.0, 1.0, 1.0], dtype=dtype))


def boost(beta: torch.Tensor) -> torch.Tensor:
    r"""Λ^μ_ν(β) along x for a batch of speeds: (...,) → (..., 4, 4)."""
    beta = torch.as_tensor(beta, dtype=torch.float64)
    g = gamma(beta)
    Lam = torch.eye(4, dtype=torch.float64).expand(*beta.shape, 4, 4).clone()
    Lam[..., 0, 0] = g
    Lam[..., 1, 1] = g
    Lam[..., 0, 1] = -g * beta
    Lam[..., 1, 0] = -g * beta
    return Lam


def transform(Lam: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    r"""x'^μ = Λ^μ_ν x^ν (ν summed).   Lam: (..., 4, 4), x: (K, 4) → (..., K, 4)"""
    return torch.einsum("...mn,kn->...km", Lam, x)


def interval(dx: torch.Tensor) -> torch.Tensor:
    r"""Δs² = η_{μν} Δx^μ Δx^ν (μ, ν summed).   dx: (..., 4) → (...)"""
    return torch.einsum("mn,...m,...n->...", metric(dx.dtype), dx, dx)


def velocity_addition(u: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """w = (u + v)/(1 + uv): the speed of v seen from a frame moving at −u (broadcasts)."""
    return (u + v) / (1 + u * v)


def four_momentum(m: torch.Tensor, beta: torch.Tensor) -> torch.Tensor:
    """p^μ = (γm, γmβ, 0, 0) in MeV (broadcast m against beta): (..., 4)."""
    g = gamma(beta)
    E, p = torch.broadcast_tensors(g * m, g * m * beta)
    zero = torch.zeros_like(E)
    return torch.stack([E, p, zero, zero], dim=-1)


def pulse_energies(L: float, beta: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    """Energy of the two light pulses (L/2 each at angles θ and θ + π in the rest frame) seen from a frame moving
    at β (relativistic Doppler factor γ(1 − β cos θ)): (..., 2)."""
    angles = torch.stack([theta, theta + torch.pi], dim=-1)  # (..., 2)
    return 0.5 * L * gamma(beta)[..., None] * (1 - beta[..., None] * torch.cos(angles))


def binding_energy(Z: torch.Tensor, N: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    """B = (Z m_H + N m_n − M) c² in MeV from atomic masses in u: (K,)."""
    return (Z * M_H + N * M_N - M) * U_MEV


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/astro/einstein_1905_relativity/tensors.py
"""Build the concrete tensors: clock speeds, nuclear masses and the Frisch & Smith (1963) muon counts."""

import csv
from pathlib import Path

import torch

DATA = Path(__file__).parent / "data" / "frisch_smith_1963_muons.csv"
C = 299.792458  # m/μs
M_ELECTRON = 0.51099895069  # MeV (CODATA 2022)
M_MUON = 105.6583755  # MeV
TAU_MUON = 2.1969811  # μs, muon lifetime at rest (PDG)

# Atomic masses in u (AME2020, Wang et al., Chinese Phys. C 45, 030003 (2021)): name → (Z, N, M)
NUCLIDES = {
    "H-2": (1, 1, 2.014101777844),
    "He-4": (2, 2, 4.002603254130),
    "Li-7": (3, 4, 7.016003434),
    "C-12": (6, 6, 12.0),
    "O-16": (8, 8, 15.994914619257),
    "Fe-56": (26, 30, 55.934936326),
    "U-238": (92, 146, 238.050786936),
}


def clock_speeds(beta_max: float, n: int) -> torch.Tensor:
    """Speeds (B,) of the light clocks, from a clock at rest up to beta_max."""
    return torch.linspace(0.0, beta_max, n, dtype=torch.float64)


def nuclear_masses(names: list[str]) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Proton numbers, neutron numbers and atomic masses (u) of the named nuclides: three (K,) tensors."""
    Z, N, M = zip(*(NUCLIDES[n] for n in names))
    return (
        torch.tensor(Z, dtype=torch.float64),
        torch.tensor(N, dtype=torch.float64),
        torch.tensor(M, dtype=torch.float64),
    )


def frisch_smith() -> dict:
    """Muon rates on Mt Washington and at sea level (Frisch & Smith 1963), with the published speed bands."""
    with DATA.open(encoding="utf-8") as f:
        rows = list(csv.DictReader(line for line in f if not line.startswith("#")))

    def col(name):
        return torch.tensor([float(r[name]) for r in rows], dtype=torch.float64)

    return {
        "station": [r["station"] for r in rows],
        "altitude": col("altitude_m"),  # (2,) m
        "rate": col("rate_per_hour"),  # (2,)
        "err": col("rate_err_per_hour"),  # (2,)
        "beta_min": col("beta_min"),
        "beta_max": col("beta_max"),
    }


def atmosphere(z: torch.Tensor, rho0: float = 1.225e-3, H: float = 8400.0) -> torch.Tensor:
    """Air column (g/cm²) above sea level up to height z (m), isothermal atmosphere ρ = ρ0 e^{−z/H} (g/cm³)."""
    return rho0 * H * 100 * (1 - torch.exp(-z / H))


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/astro/einstein_1905_relativity/solve.py
"""Light clocks, Einstein's rod and train, the 1905 mass-loss argument, and the γ of Frisch & Smith's muons."""

import torch

from . import equation, tensors


def light_clocks(beta: torch.Tensor, D: float, t_max: float, n: int):
    """Photon paths in the lab for clocks moving at speeds beta (B,): times (T,), x and y (B, T).

    The photon moves at c = 1 along a zigzag: horizontal speed β (it rides with the clock), vertical speed √(1 − β²).
    """
    t = torch.linspace(0.0, t_max, n, dtype=torch.float64)
    x = beta[:, None] * t  # (B, T)
    phase = torch.remainder(t * torch.sqrt(1 - beta[:, None] ** 2) / D, 2.0)  # 0 → 1 up, 1 → 2 down
    y = D * (1 - (phase - 1).abs())
    return t, x, y


def count_ticks(y: torch.Tensor) -> torch.Tensor:
    """Completed round trips (returns to the bottom mirror) along the last axis: (B,)."""
    dy = y.diff(dim=-1)
    return ((dy[..., :-1] < 0) & (dy[..., 1:] >= 0)).sum(dim=-1)


def contracted_length(L0: float, beta: torch.Tensor) -> torch.Tensor:
    """Length L of a clock lying along the motion that ticks together with an identical clock standing across it.

    Across: 2 L0 γ (the light-clock diagonal). Along, length L: L/(1 − β) + L/(1 + β) = 2 L γ². Equal ⇒ L = L0/γ.
    """
    g = equation.gamma(beta)
    across = 2 * L0 * g
    along_per_length = 1 / (1 - beta) + 1 / (1 + beta)
    return across / along_per_length


def kinetic_energy_drop(L: float, beta: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    """Einstein (1905): K0 − K1 = (energy of the light in the moving frame) − (in the rest frame) = γL − L."""
    return equation.pulse_energies(L, beta, theta).sum(dim=-1) - L


def annihilate_at_rest(m: float) -> torch.Tensor:
    """e⁺e⁻ at rest → two back-to-back photons: (2, 4) four-momenta (E, p_x, p_y, p_z) in MeV."""
    return torch.tensor([[m, m, 0.0, 0.0], [m, -m, 0.0, 0.0]], dtype=torch.float64)


def muon_flight(data: dict, dEdx: float, n_speeds: int = 41, n_z: int = 2001) -> dict:
    """Muons fly from the summit down to sea level, slowing as they ionise the air: lab and proper flight times.

    Batched over the published speed band at the summit (B,) and over heights (Z,).
    """
    z_top, z_bot = data["altitude"]
    beta_top = torch.linspace(float(data["beta_min"][0]), float(data["beta_max"][0]), n_speeds, dtype=torch.float64)
    z = torch.linspace(float(z_top), float(z_bot), n_z, dtype=torch.float64)  # (Z,) going down
    column = tensors.atmosphere(z_top) - tensors.atmosphere(z)  # (Z,) g/cm² crossed so far
    E = tensors.M_MUON * equation.gamma(beta_top)[:, None] - dEdx * column  # (B, Z) MeV
    g = E / tensors.M_MUON
    beta = torch.sqrt(1 - 1 / g**2)
    s = z_top - z  # (Z,) distance flown so far, m
    t_lab = torch.trapezoid(1 / (beta * tensors.C), s, dim=-1)  # (B,) μs: ∫ dz / v
    tau = torch.trapezoid(1 / (beta * g * tensors.C), s, dim=-1)  # (B,) μs: ∫ dz / (γ v), the muon's own clock
    return {"z": z, "gamma": g, "beta": beta, "t_lab": t_lab, "tau": tau}


def muon_gamma(data: dict, dEdx: float = 1.82) -> dict:
    """Fit the time-dilation factor γ to the two count rates (autograd Newton steps on χ²), compare with the speeds."""
    flight = muon_flight(data, dEdx)
    t_lab = flight["t_lab"].mean()
    gamma_pred = float(t_lab / flight["tau"].mean())  # dilation the measured speeds predict

    N_top, N_bot = data["rate"]
    r = N_bot / N_top  # surviving fraction
    sigma_r = r * torch.sqrt((data["err"] / data["rate"]).pow(2).sum())

    def chi2(g):
        return ((r - torch.exp(-t_lab / (g * tensors.TAU_MUON))) / sigma_r) ** 2

    g = torch.tensor(5.0, dtype=torch.float64, requires_grad=True)
    for _ in range(30):  # Newton's method with autograd derivatives
        (d1,) = torch.autograd.grad(chi2(g), g, create_graph=True)
        (d2,) = torch.autograd.grad(d1, g)
        with torch.no_grad():
            g -= d1 / d2
    (d1,) = torch.autograd.grad(chi2(g), g, create_graph=True)
    (d2,) = torch.autograd.grad(d1, g)
    return {
        "gamma_fit": float(g.detach()),
        "gamma_err": float(torch.sqrt(2 / d2)),  # Δχ² = 1
        "gamma_pred": gamma_pred,
        "t_lab": float(t_lab),
        "tau_rest": float(t_lab) / float(g.detach()),
        "no_dilation_rate": float(N_top * torch.exp(-t_lab / tensors.TAU_MUON)),
        "beta_bottom": (float(flight["beta"][:, -1].min()), float(flight["beta"][:, -1].max())),
        "flight": flight,
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/astro/einstein_1905_relativity/plot.py
"""Light clocks racing along, γ(β) with Frisch & Smith's muons, and binding energies from E = mc²."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from . import equation

AMBER, GREY = "#f59e0b", "#9ca3af"
ROW = 1.6  # vertical spacing of the clocks


def animate(beta, t, x, y, nuclei, mu, path: Path) -> Path:
    beta_np, t_np, x_np, y_np = beta.numpy(), t.numpy(), x.numpy(), y.numpy()
    B = len(beta_np)
    gam = equation.gamma(beta).numpy()
    names, Bind, A = (v.numpy() if isinstance(v, torch.Tensor) else v for v in nuclei)

    fig, (ax_c, ax_g, ax_m) = plt.subplots(1, 3, figsize=(11, 4.3), gridspec_kw={"width_ratios": [1.35, 1, 1]})

    # --- light clocks
    ax_c.set_xlim(-0.4, t_np[-1] + 0.4)
    ax_c.set_ylim(-0.4, ROW * (B - 1) + 1.5)
    ax_c.set_xlabel("lab position $x$ (mirror spacings, $c = 1$)")
    ax_c.set_yticks([])
    ax_c.grid(False)
    rows = []
    for i in range(B):
        y0 = ROW * i
        rows.append(
            {
                "y0": y0,
                "mirrors": [ax_c.plot([], [], color="#374151", lw=2.5)[0] for _ in range(2)],
                "trail": ax_c.plot([], [], color=AMBER, lw=1.2)[0],
                "photon": ax_c.plot([], [], "o", color=AMBER, ms=5)[0],
                "label": ax_c.text(-0.3, y0 + 1.15, "", fontsize=7.5, color="#374151"),
            }
        )

    # --- γ(β), length contraction and the muons
    one_minus = np.logspace(0, -3.3, 600)  # 1 − β, so speeds near c get room
    b = 1 - one_minus
    ax_g.plot(one_minus, 1 / np.sqrt(1 - b**2), color=ACCENT[0], lw=1.8, label="time dilation $\\gamma$")
    ax_g.plot(one_minus, np.sqrt(1 - b**2), color=ACCENT[2], lw=1.5, label="length $L/L_0 = 1/\\gamma$")
    ax_g.plot(1 - beta_np, gam, "o", color=AMBER, ms=6, zorder=3, label="the clocks")
    beta_mu = float(mu["flight"]["beta"].mean())
    ax_g.errorbar(
        [1 - beta_mu],
        [mu["gamma_fit"]],
        yerr=[mu["gamma_err"]],
        fmt="s",
        color=ACCENT[1],
        ms=6,
        capsize=3,
        label=f"muon counts: {mu['gamma_fit']:.1f} ± {mu['gamma_err']:.1f}",
    )
    ax_g.plot(
        [1 - beta_mu], [mu["gamma_pred"]], "D", color=ACCENT[3], ms=5, label=f"muon speeds: {mu['gamma_pred']:.1f}"
    )
    ax_g.set_yscale("log")
    ax_g.set_xscale("log")
    ax_g.set_xlim(1.0, 5e-4)  # reversed: faster to the right
    ax_g.set_ylim(0.03, 40)
    ax_g.set_xlabel("$1 - v/c$  (faster →)")
    ax_g.set_title(
        "Frisch & Smith 1963: 563/h on Mt Washington,\n408/h at sea level (30/h without dilation)", fontsize=8
    )
    ax_g.legend(fontsize=6.5, loc="upper left")

    # --- E = mc²: binding energy per nucleon from atomic masses
    per = Bind / A
    bars = ax_m.bar(range(len(names)), per, color=ACCENT[0], alpha=0.8)
    he = list(names).index("He-4")
    bars[he].set_color(AMBER)
    ax_m.set_xticks(range(len(names)), names, fontsize=7, rotation=40)
    ax_m.set_ylabel("binding energy per nucleon (MeV)")
    ax_m.set_ylim(0, 10.5)
    ax_m.text(he, per[he] + 0.3, f"{Bind[he]:.2f} MeV\n= 0.0304 u·c²", ha="center", fontsize=7, color="#b45309")
    ax_m.set_title("$E = mc^2$: missing mass $Zm_H + Nm_n - M$\ne⁺e⁻ → 2 × 0.511 MeV photons", fontsize=8)

    footer = fig.text(0.5, 0.01, "", ha="center", fontsize=8.5)

    def update(k: int):
        for i, r in enumerate(rows):
            xc = x_np[i, k]
            r["mirrors"][0].set_data([xc - 0.3, xc + 0.3], [r["y0"], r["y0"]])
            r["mirrors"][1].set_data([xc - 0.3, xc + 0.3], [r["y0"] + 1, r["y0"] + 1])
            r["trail"].set_data(x_np[i, : k + 1], r["y0"] + y_np[i, : k + 1])
            r["photon"].set_data([xc], [r["y0"] + y_np[i, k]])
            n_ticks = int(np.sum((np.diff(y_np[i, : k + 1])[:-1] < 0) & (np.diff(y_np[i, : k + 1])[1:] >= 0)))
            r["label"].set_text(f"β = {beta_np[i]:.2f}   γ = {gam[i]:.2f}   ticks: {n_ticks}")
        ax_c.set_title(f"light clocks, lab time t = {t_np[k]:.1f}: one tick takes γ × 2D/c", fontsize=9)
        footer.set_text(
            "Postulates: same laws in every inertial frame, light always at c  ⇒  the photon's diagonal path makes"
            " moving clocks slow by γ, rods short by 1/γ"
        )

    update(0)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    frames = frame_indices(len(t_np), 80) + [len(t_np) - 1] * 10
    return save_gif(fig, update, frames, path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'beta_fast': 0.95,
    'n_clocks': 4,
    'dEdx': 1.82
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.astro.einstein_1905_relativity")]:
    del sys.modules[name]
main = importlib.import_module("problems.astro.einstein_1905_relativity.main")
problem_dir = Path("problems/astro/einstein_1905_relativity")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
